In [1]:
import time
import pickle
import random
import os
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

load_dotenv();

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/104.0.5112.79 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Version/15.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/104.0.5112.79 Safari/537.36"
]

In [2]:
def setupDriver():
    # Set up Brave options
    options = Options()
    options.binary_location = "/usr/bin/brave-browser"  # Update with your Brave installation path

    # Choose a random User-Agent from the list
    user_agent = random.choice(USER_AGENTS)
    options.add_argument(f"user-agent={user_agent}")
    options.add_argument("--disable-blink-features=AutomationControlled")  # Helps evade bot detection

    # Define the user data directory where cookies will be stored
    user_data_dir = os.path.join(os.getcwd(), "cookies")  # Create a 'cookies' directory in the current working directory
    options.add_argument(f"user-data-dir={user_data_dir}")  # Use this directory for storing cookies

    # Create a WebDriver instance using the Brave browser (ensure ChromeDriver is in PATH)
    driver = webdriver.Chrome(options=options)
    return driver

In [3]:
# Function to check if the account menu exists, indicating a successful login
def checkAccountLoggedIn(driver):
    if driver.current_url == "https://x.com/home":
        print("Check Account Logged in = true")
        return True
    print("Check Account Logged in = false")
    return False

In [4]:
# Function to perform login (if needed) and handle cookies
def login():
    driver = setupDriver()

    # Try to load cookies from previous session if they exist
    try:
        driver.get("https://x.com")  # Navigate to the homepage or login page to check if we have cookies
        time.sleep(10)  # Wait for the page to load
        
        # Check if the account menu is available, meaning we're already logged in
        if checkAccountLoggedIn(driver):
            print("Logged in using cookies.")
            return driver  # Return the driver with cookies applied
        else:
            print("Cookies did not work. Logging in manually...")
            raise Exception("Cookies invalid or expired, logging in manually.")  # Force manual login

    except Exception as e:
        print(f"Error: {e}")
        print("Logging in manually...")
        driver.get("https://x.com/i/flow/login")
        time.sleep(10)  # Wait for login page to load

        # Manually log in (provide your credentials here)
        username_field = driver.find_element(By.NAME, "text")
        username_field.send_keys(os.getenv("twitterEmail"))
        driver.find_element(By.XPATH, "//span[text()='Next']").click()
        time.sleep(6)

        password_field = driver.find_element(By.NAME, "password")
        password_field.send_keys(os.getenv("twitterPassword"))
        driver.find_element(By.XPATH, "//span[text()='Log in']").click()
        time.sleep(7)

        print("Successfully Logged in")
        return driver  # Return the logged-in driver

In [ ]:
def extract_tweets(driver):
    tweets = []  # Initialize an empty list to store tweet data
    
    try:
        # Extract tweet containers (articles with data-testid="tweet")
        tweetElements = driver.find_elements(By.XPATH, "//article[@data-testid='tweet']")
        
        for tweet in tweetElements:
            try:
                # Get the div with tweetText (multiple spans with tweet content)
                tweetTextElement = tweet.find_element(By.XPATH, ".//div[@data-testid='tweetText']")
                
                # Get the text from all the span elements inside the tweetText div
                tweet_spans = tweetTextElement.find_elements(By.XPATH, ".//span")
                tweet_text = " ".join([span.text for span in tweet_spans])  # Join all span texts into a single string
                
                # Get tweet metadata (reply count, retweet count, like count, view count)
                reply_count = tweet.find_element(By.XPATH, ".//div[@data-testid='reply']").text
                retweet_count = tweet.find_element(By.XPATH, ".//div[@data-testid='retweet']").text
                like_count = tweet.find_element(By.XPATH, ".//div[@data-testid='like']").text
                view_count = tweet.find_element(By.XPATH, ".//div[@data-testid='view']").text
                
                # Get tweet creation date (timestamp)
                created_date = tweet.find_element(By.XPATH, ".//time").get_attribute("datetime")
                
                # Store tweet data in a dictionary
                tweet_data = {
                    "text": tweet_text,
                    "reply_count": reply_count,
                    "retweet_count": retweet_count,
                    "like_count": like_count,
                    "view_count": view_count,
                    "created_date": created_date
                }
                
                tweets.append(tweet_data)  # Add the tweet data to the list of tweets
                
            except Exception as e:
                print(f"Error extracting data from tweet: {e}")
        
    except Exception as e:
        print(f"Error while extracting tweets: {e}")
    
    return tweets


In [5]:
def tempLogin():
    driver = setupDriver()
    #driver.get("https://x.com/i/flow/login")

    driver.get("https://x.com")  # Navigate to the homepage or login page to check if we have cookies
    time.sleep(5)  # Wait for the page to load
        
    # Check if the account menu is available, meaning we're already logged in
    if checkAccountLoggedIn(driver):
        print("Logged in using cookies.")
        return driver  # Return the driver with cookies applied

In [ ]:
driver = login()
# driver = setupDriver()
# driver.get("https://x.com")

Check Account Logged in = true
Logged in using cookies.


In [11]:
driver.quit()